# RSO-114: Analyze airflow behavior and overall thermal trends

During the shutdown period (31.1.26 -14.2.26) we ran multiple versions (different louver configurations) of BLOCK-T679. This notebook analyzes the thermal behavior for different airflow conditions.

**Description**

Use airflow measurements (when available) to see whether louver settings actually change ventilation, and summarize what seems to work better or worse.

**Expected results:**

Plots: airflow vs time and airflow vs temperature change

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from astropy.time import Time

from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient
from lsst.summit.utils.tmaUtils import TMAEventMaker, TMAState

In [ ]:
t_start_period = Time("2026-01-31T00:00:00Z", scale="utc")
t_end_period = Time("2026-02-15T00:00:00Z", scale="utc")

efd_client = makeEfdClient()

# Queries

In [ ]:
def query_setlouvers(start, end):
    df_louvers = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTDome.command_setLouvers",
        columns=["*"],
        begin=start,
        end=end,
    )

    return df_louvers

# Configuration of louvers

In [ ]:
df_setlouvers = query_setlouvers(t_start_period, t_end_period)

In [ ]:
# Copy current index into a new column before any merge
df_setlouvers['time_stamp'] = df_setlouvers.index

In [ ]:
# Select all columns that start with "position"
position_cols = df_setlouvers.filter(regex=r'^position').columns

# Sort columns
position_cols = sorted(position_cols, key=lambda x: int(x.replace('position', '')))

# Compute unique combinations
combination_counts = (
    df_setlouvers[position_cols]
    .value_counts()
    .reset_index(name='count')
)

# Create configuration ID (1 to N)
combination_counts['louvers_conf'] = range(1, len(combination_counts) + 1)

# Merge configuration ID back into original dataframe
df_setlouvers = df_setlouvers.merge(
    combination_counts[position_cols + ['louvers_conf']],
    on=position_cols,
    how='left'
)

print(f"Number of unique configurations detected: {len(combination_counts)}\n")

# Print configurations showing only non-zero positions
for _, row in combination_counts.iterrows():
    
    conf_id = row['louvers_conf']
    count = row['count']
    
    print(f"Configuration {conf_id} (appears {count} times):")
    
    # Extract position values
    config = row[position_cols]
    
    # Keep only non-zero values
    non_zero = config[config != 0]
    
    if len(non_zero) == 0:
        print("  All positions are 0")
    else:
        for col, val in non_zero.items():
        #for col, val in config.items():
            print(f"  {col}: {val}")
    
    print("-" * 40)

15 combinations have been made, varying the opening of the louvers: 2, 11, 12, 20, 21, and 29.

In [ ]:
df_setlouvers['duration'] = (
    df_setlouvers['time_stamp'].shift(-1) - df_setlouvers['time_stamp']
)
df_setlouvers['duration_minutes'] = df_setlouvers['duration'].dt.total_seconds() / 60

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))
ax.scatter(df_setlouvers['louvers_conf'],df_setlouvers['duration_minutes'])
ax.set_ylim(0,100)

In [ ]:
louvers_used = '[2,11,12,20,21,29]'
config_description = {1:f'{louvers_used} 100%',
                      2:f'{louvers_used} 10%', 
                      3:f'{louvers_used} 50%',
                      4:f'[11,21,29] 100%',
                      5:f'[2,11,12] 100%',
                      6:f'[20,21,29] 100%',
                      7:f'[11,12,20,21,29] 50%',
                      8:f'[2,12,20,21,29] 50% -- 11 100%',
                      9:f'[2,12,20,21,29] 50%',
                      10:f'[11,20,29] 100%',
                      11:f'{louvers_used} 30%',
                      12:f'[11,12,20,21,29] 50% -- 2 100%',
                      13:f'{louvers_used} 60%',
                      14:f'{louvers_used} 0%',
                      15:f'[11,20,29] 50%',
                     }

for cfg in range(1,16):
    df_airflow_list = []
    df_temp_m1m3_list = []
    df_temp_cam_list = []
    df_temp_m2_list = []
    df_temp_weat_list = []
    
    condition = df_setlouvers['duration_minutes'].between(20, 100, inclusive="neither") & (df_setlouvers['louvers_conf'] == cfg)
    df_selected = df_setlouvers[condition].copy()
    df_selected['end_time_stamp'] = df_selected['time_stamp'] + df_selected['duration']
    
    #retrieve data for this configuration
    print(f"Number of tests selected for configuration {cfg}:",len(df_selected))
    for row in df_selected.itertuples():
        begin = Time(row.time_stamp)
        end = Time(row.end_time_stamp)
        df_air = getEfdData(
            client=efd_client,
            topic="lsst.sal.ESS.airFlow", #seems that it comes from the weather tower
            columns=["direction","speed"],
            begin=begin,
            end=end,
        )
        df_temp = getEfdData(
            client=efd_client,
            topic="lsst.sal.ESS.temperature", 
            columns=["sensorName","temperatureItem0"],
            begin=begin,
            end=end,
        )
        df_airflow_list.append(df_air)
        dft_m1m3 = df_temp[df_temp["sensorName"]=='M1M3-ESS03'].copy()
        dft_cam = df_temp[df_temp["sensorName"]=='Camera-ESS01'].copy()
        dft_m2 = df_temp[df_temp["sensorName"]=='M2-ESS02'].copy()
        dft_weat = df_temp[df_temp["sensorName"]=='Weather tower air temperature'].copy()
        df_temp_m1m3_list.append(dft_m1m3)
        df_temp_cam_list.append(dft_cam)
        df_temp_m2_list.append(dft_m2)
        df_temp_weat_list.append(dft_weat)
    
    #and now make plots
    for i,(dfa,dft_m1m3,dft_m2,dft_cam,dft_weat) in enumerate(zip(df_airflow_list,df_temp_m1m3_list,df_temp_m2_list,df_temp_cam_list,df_temp_weat_list)):
        fig, ax1 = plt.subplots()
        t0 = dfa.index[0]
        t_air = (dfa.index - t0).total_seconds()
        t_m1m3_temp = (dft_m1m3.index - t0).total_seconds()
        t_m2_temp = (dft_m2.index - t0).total_seconds()
        t_cam_temp = (dft_cam.index - t0).total_seconds()
        t_weat_temp = (dft_weat.index - t0).total_seconds()
        ax1.scatter(t_air, dfa["speed"],marker='.',label='Air flow speed',color='blue')
        ax1.set_xlabel("Time since start (s)")
        ax1.set_ylabel("Airflow speed (m/s)")
        ax2 = ax1.twinx()
        ax2.scatter(t_m1m3_temp, dft_m1m3["temperatureItem0"],label='M1M3-ESS03 T',color='red')
        ax2.scatter(t_weat_temp, dft_weat["temperatureItem0"],label='Weather tower T',color='orange')
        ax2.scatter(t_cam_temp, dft_cam["temperatureItem0"],label='Camera-ES01 T',color='yellow')
        ax2.set_ylabel("Temperature (C)")
        fig.suptitle(f"Config. {cfg} -- {config_description[cfg]}")
        fig.legend()
        plt.savefig(f"plots/config_{cfg}_case_{i}.png")
    plt.close()
    #plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))
ax.plot(df_temp0.iloc[::100])
for i,t in enumerate(df_setlouvers['time_stamp']):
    if i==0:
        ax.axvline(t, color='red', alpha=0.3, label='Louver config set')
    else:
        ax.axvline(t, color='red', alpha=0.3)
ax.set_xlabel("Time")
ax.set_ylabel("Temperature sensor 0")
ax.set_title("Temperature with Louver Configurations")
ax.legend()
